In [2]:
import numpy as np
import pandas as pd

In [3]:
df=pd.read_csv("Data_Train_Cleaned.csv")
df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302


## What is Feature Engineering?

> Feature engineering means creating better input features from the existing data so the model can discover patterns more easily.

**Raw Data**

```Dep_Time = 22:20```

↓

**Engineered Features**


```
Departure_Hour = 22
Departure_Minute = 20
Is_Night = 1
```


> The second version contains much more useful information.

### Step 1: Date Features

*Current feature*

`Date_of_Journey` - `24/03/2019`



*Extract*

In [4]:
df['Date_of_Journey']=pd.to_datetime(df['Date_of_Journey'],format="%d/%m/%Y")

df['Journey_Day'] = df['Date_of_Journey'].dt.day
df['Journey_Month'] = df['Date_of_Journey'].dt.month
df['Journey_Weekday'] = df['Date_of_Journey'].dt.day_name()
df['Journey_DayOfWeek'] = df['Date_of_Journey'].dt.dayofweek

df["Is_Weekend"] = df["Journey_DayOfWeek"].isin([5,6]).astype(int) #because ticket prices often change on weekends.

### Droping the main column because we dont want it 

In [5]:
df.drop('Date_of_Journey', axis=1, inplace=True)

### Step 2: Departure Time

*Current feature*

`Dep_Time` - `22:20`


*Extract*

In [6]:
df['Dep_Time']=pd.to_datetime(df['Dep_Time'], format='%H:%M')

df['Dep_Hour']=df['Dep_Time'].dt.hour 
df['Dep_Minute']=df['Dep_Time'].dt.minute

### Use this if required

```Python
def part_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df["Dep_Period"] = df["Dep_Hour"].apply(part_of_day) #Flights leaving at night often have different prices.
```

In [7]:
df.drop('Dep_Time', axis=1, inplace=True)

### Step 3: Arrival Time

Same Idea

In [8]:
df["Arrival_Time"] = df["Arrival_Time"].str.split().str[0]

df["Arrival_Time"] = pd.to_datetime(df["Arrival_Time"], format='%H:%M')

df["Arrival_Hour"] = df["Arrival_Time"].dt.hour
df["Arrival_Minute"] = df["Arrival_Time"].dt.minute

In [9]:
df.drop('Arrival_Time', axis=1, inplace=True)

### Step 4: Duration

*Current* -  `2h 50m`

**Convert to minutes**

In [10]:
hours= df['Duration'].str.extract(r"(\d+)h")[0].fillna(0).astype(int)
minutes= df['Duration'].str.extract(r"(\d+)m")[0].fillna(0).astype(int)

df['Duration_Minutes'] =  hours * 60 + minutes

In [11]:
df.drop('Duration', axis=1, inplace=True)

### Step 5: Total Stops

*Current*

```
non-stop
1 stop
2 stops
```

*Convert*

In [12]:
df['Total_Stops'].unique()

array(['non-stop', '2 stops', '1 stop', '3 stops', '4 stops'],
      dtype=object)

In [13]:
df['Total_Stops']=df['Total_Stops'].replace(
    {
        'non-stop' : 0,
        '1 stop' : 1,
        '2 stops' : 2,
        '3 stops' : 3,
        '4 stops' : 4
    }
)

C:\Users\dhara\AppData\Local\Temp\ipykernel_8624\557842064.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Total_Stops']=df['Total_Stops'].replace(


### Step 6: Route

`Current` --  `Delhi → Mumbai → Cochin`

> Instead of treating the entire route as one category, split it.

In [14]:
df["Number_of_Airports"] = df["Route"].str.split("→").str.len()

### Step 7: Additional Info

> We check the frequencies of the value and then decide whether to drop it or keep it

In [24]:
summary = pd.DataFrame({
    'Count' : df['Additional_Info'].value_counts() ,
    'Frequency' :  df['Additional_Info'].value_counts(normalize=True).mul(100).round(2)
})

summary

,Count,Frequency
Additional_Info,,
No info,8344,78.11
In-flight meal not included,1982,18.55
No check-in baggage included,320,3.00
1 Long layover,19,0.18
Change airports,7,0.07
Business class,4,0.04
No Info,3,0.03
1 Short layover,1,0.01
Red-eye flight,1,0.01


In [26]:
#Since most of the values are No info, it contributes very very less to the predictive model. So, it's better to drop this!!

df.drop('Additional_Info', axis=1, inplace=True)

In [27]:
df.to_csv('Data_Train_Feature_Engineered.csv', index=False)